In [7]:
from google.colab import drive


drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
pip install scikit-image==0.19.3

  Using cached scikit-image-0.19.3.tar.gz (22.2 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [23]:
import skimage.feature
dir(skimage.feature)

['BRIEF',
 'CENSURE',
 'Cascade',
 'ORB',
 'SIFT',
 'blob_dog',
 'blob_doh',
 'blob_log',
 'canny',
 'corner_fast',
 'corner_foerstner',
 'corner_harris',
 'corner_kitchen_rosenfeld',
 'corner_moravec',
 'corner_orientations',
 'corner_peaks',
 'corner_shi_tomasi',
 'corner_subpix',
 'daisy',
 'draw_haar_like_feature',
 'draw_multiblock_lbp',
 'fisher_vector',
 'graycomatrix',
 'graycoprops',
 'haar_like_feature',
 'haar_like_feature_coord',
 'hessian_matrix',
 'hessian_matrix_det',
 'hessian_matrix_eigvals',
 'hog',
 'learn_gmm',
 'local_binary_pattern',
 'match_descriptors',
 'match_template',
 'multiblock_lbp',
 'multiscale_basic_features',
 'peak_local_max',
 'plot_matched_features',
 'shape_index',
 'structure_tensor',
 'structure_tensor_eigenvalues']

In [24]:
import skimage.feature
print([name for name in dir(skimage.feature) if "grey" in name])

[]


In [27]:
import skimage.feature.texture
print(dir(skimage.feature.texture))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_glcm_loop', '_local_binary_pattern', '_multiblock_lbp', 'check_nD', 'draw_multiblock_lbp', 'gray2rgb', 'graycomatrix', 'graycoprops', 'img_as_float', 'local_binary_pattern', 'multiblock_lbp', 'np', 'warnings']


In [29]:
pip install mahotas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 62.3 MB/s eta 0:00:00


In [31]:
import os
import glob
import numpy as np
import pandas as pd
from skimage.io import imread

In [32]:
def greycomatrix_manual(image, distances=[1], angles=[0], levels=256, symmetric=True, normed=True):
    rows, cols = image.shape
    glcms = []
    for d in distances:
        for angle in angles:
            dx = int(np.round(np.cos(angle)))
            dy = int(np.round(np.sin(angle)))
            glcm = np.zeros((levels, levels), dtype=np.float64)
            for i in range(rows):
                for j in range(cols):
                    x = i + dy * d
                    y = j + dx * d
                    if 0 <= x < rows and 0 <= y < cols:
                        p1 = image[i, j]
                        p2 = image[x, y]
                        glcm[p1, p2] += 1
                        if symmetric:
                            glcm[p2, p1] += 1
            if normed:
                glcm = glcm / glcm.sum()
            glcms.append(glcm)
    return np.array(glcms)

def greycoprops_manual(glcm, prop):
    i, j = np.indices(glcm.shape)
    if prop == 'contrast':
        return np.sum(glcm * (i - j)**2)
    elif prop == 'homogeneity':
        return np.sum(glcm / (1.0 + np.abs(i - j)))
    elif prop == 'energy':
        return np.sum(glcm**2)
    elif prop == 'ASM':
        return np.sum(glcm**2)
    elif prop == 'entropy':
        return -np.sum(glcm * np.log2(glcm + 1e-10))
    elif prop == 'correlation':
        mean_i = np.sum(i * glcm)
        mean_j = np.sum(j * glcm)
        std_i = np.sqrt(np.sum(glcm * (i - mean_i)**2))
        std_j = np.sqrt(np.sum(glcm * (j - mean_j)**2))
        return np.sum(glcm * (i - mean_i) * (j - mean_j)) / (std_i * std_j + 1e-10)
    else:
        raise ValueError("Property tidak dikenal")

In [33]:
folder_path = "/content/drive/MyDrive/Colab Notebooks/Dataset"
files = glob.glob(os.path.join(folder_path, "*.jpg"))[:10]

data = []

for f in files:
    image = imread(f, as_gray=True)
    image = (image * 255).astype(np.uint8)

    glcm = greycomatrix_manual(image, distances=[1], angles=[0], levels=256)
    g = glcm[0]

    features = {
        "filename": os.path.basename(f),
        "contrast": greycoprops_manual(g, 'contrast'),
        "homogeneity": greycoprops_manual(g, 'homogeneity'),
        "energy": greycoprops_manual(g, 'energy'),
        "ASM": greycoprops_manual(g, 'ASM'),
        "entropy": greycoprops_manual(g, 'entropy'),
        "correlation": greycoprops_manual(g, 'correlation')
    }
    data.append(features)

In [34]:
df = pd.DataFrame(data)
df.to_excel("Hasi_10Datasetglcm.xlsx", index=False)

print("Ekstraksi selesai, hasil disimpan di glcm_features.xlsx")

Ekstraksi selesai, hasil disimpan di glcm_features.xlsx


In [35]:
import os
os.listdir()

['.config', 'drive', 'Hasi_10Datasetglcm.xlsx', 'sample_data']

In [37]:
from google.colab import files
files.download("Hasi_10Datasetglcm.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>